# Fake Jobs – Semantisch (PCA 30 & 100)
- PCA über den Embedding-Block, **nur auf Trainingszeilen gefittet**, dann auf alle Zeilen angewendet
- Erklärte Varianz wird je Variante geprintet (Ziel > 80 %)

In [ ]:
import os
import pandas as pd
from sklearn.decomposition import PCA

SEED = int(os.environ.get("SEED", 1))
print("SEED", SEED)

## Basis, Embeddings & Split laden
- Numerischer Block kommt aus dem seed-spezifischen `cleaned`, die Embeddings sind seed-unabhängig

In [ ]:
base = pd.read_csv(f"../../data/preprocessed/cleaned_fake_jobs_seed{SEED}.csv")
emb = pd.read_parquet("../../data/preprocessed/semantic_emb_fake_jobs.parquet")
split = pd.read_csv(f"../../data/splits/split_fake_jobs_seed{SEED}.csv")

df = base.merge(emb, on="row_id", how="inner")
emb_cols = [c for c in df.columns if "_emb_" in c]
other_cols = [c for c in df.columns if c not in emb_cols]
tr = df["row_id"].isin(split.loc[split["split"] == "train", "row_id"]).values

# the non-embedding block must be exactly the cleaned representation
assert other_cols == list(base.columns)
print("Zeilen:", len(df), "| Embedding-Spalten:", len(emb_cols), "| Train:", int(tr.sum()))

## PCA 30 & 100 – auf Train gefittet, speichern

In [ ]:
for n in (30, 100):
    pca = PCA(n_components=n, random_state=SEED).fit(df.loc[tr, emb_cols])
    red = pca.transform(df[emb_cols])
    print(f"explained variance ({n} comps):", round(pca.explained_variance_ratio_.sum(), 4))
    out = pd.concat([df[other_cols].reset_index(drop=True),
                     pd.DataFrame(red, columns=[f"pca_{i}" for i in range(n)])], axis=1)
    out.to_csv(f"../../data/preprocessed/semantic_pca{n}_fake_jobs_seed{SEED}.csv", index=False)
    print("gespeichert:", out.shape)

## Verifikation

In [ ]:
assert out.isna().sum().sum() == 0
assert out["row_id"].is_unique